# GMM（高斯混合模型，Gaussian Mixture Model）

- **核心原理**：EM 演算法、軟分群
- **變體**：共變異數的四種形式（full、tied、diag、spherical）；Bayesian GMM（自動決定要用幾個成分）
- **推薦資料集**：Iris（150 朵鳶尾花、4 個量測（cm）、3 個品種；品種只用於評估）

第十三章 K-Means 的兩個限制：

1. **硬分群**：每個點 100% 屬於某一群，即使它剛好落在兩群中間；
2. **每一群都是「圓的、大小差不多」**：用歐氏距離找最近中心，等於假設每群是同樣大小的球。

GMM 換成**機率模型**：假設資料是從 $K$ 個高斯分布（每個可以是任意方向、任意大小的**橢圓**）混在一起產生的。分群 = 反推「每個點是從哪個高斯來的」**機率**（軟分群）。參數用 **EM 演算法**估計：

- **E 步**：用目前的參數，算每個點屬於每一群的機率（責任，responsibility）—— 就是第六章樸素貝氏的貝氏定理；
- **M 步**：用這些機率當權重，重新估計每一群的權重、平均、共變異數 —— 就是「加權版」的平均與共變異數。

這一章的重點：**多變量常態分布的 log 密度怎麼穩定地算、EM 為什麼保證 log-likelihood 不降、GMM 跟 K-Means 的關係、共變異數的形式怎麼選（BIC）。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：4 個特徵 `sepal length`、`sepal width`、`petal length`、`petal width`（cm，單位相同、所以不標準化）。品種 0 = setosa、1 = versicolor、2 = virginica，**只用於評估**。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$、$d$ | 樣本數、特徵數 | 150、4 |
| $K$ | 成分（群）數 | — |
| $\pi_k$ | 第 $k$ 個成分的權重（混合比例），$\sum_k \pi_k = 1$ | $(K,)$ |
| $\mu_k$ | 第 $k$ 個成分的平均 | $(K, d)$ |
| $\Sigma_k$ | 第 $k$ 個成分的共變異數矩陣（對稱、正定） | $(K, d, d)$ |
| $\mathcal{N}(x; \mu, \Sigma)$ | 多變量常態分布的密度 | — |
| $r_{ik}$ | 責任：$P(z_i = k \mid x_i)$，第 $i$ 點來自第 $k$ 個成分的機率；每列總和 1 | $(n, K)$ |
| $N_k$ | $\sum_i r_{ik}$：第 $k$ 群的「有效點數」 | $(K,)$ |
| $\ell$ | 平均 log-likelihood $\frac{1}{n}\sum_i \log \sum_k \pi_k \mathcal{N}(x_i; \mu_k, \Sigma_k)$ | 純量 |
| BIC | $-2 n \ell + p \log n$（$p$ = 參數個數）；越小越好 | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第六章 | 樸素貝氏（`01-classic-ml/06-naive-bayes`） | 貝氏定理 $P(y \mid x) \propto P(x \mid y) P(y)$；在 log 空間算 JLL、用 log-sum-exp 換成機率；Gaussian NB 假設每類每個特徵**各自**是常態（共變異數是對角的）。 |
| 第六章 TODO 6 | 同上 | $\log P(y=k \mid x) = \text{JLL}_k - \text{logsumexp}(\text{JLL})$；不能先 exp 再正規化（underflow）。 |
| 第六章 TODO 3 | 同上 | `Y.T @ X` = 每一類的特徵總和（one-hot 的分組加總）；本章把 one-hot 換成「機率」$R$。 |
| 第十三章 | K-Means | Lloyd：分配 → 更新，坐標下降，目標單調不增；初始值會影響結果；`clustering_metrics`、ARI。 |
| 第十四章 | DBSCAN | 依密度、任意形狀、有雜訊；不同密度的群很難處理。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 共變異數矩陣**：$\Sigma = \frac{1}{n}\sum_i (x_i - \mu)(x_i - \mu)^\top$，$(d, d)$。對角線是各特徵的變異數，非對角線是兩兩的共變異數。例：二維點 $(0, 0), (2, 2)$ → $\mu = (1, 1)$、$\Sigma = \begin{pmatrix}1 & 1\\ 1 & 1\end{pmatrix}$（兩個特徵完全一起變）。

**2. 多變量常態分布**：
$$\log \mathcal{N}(x; \mu, \Sigma) = -\frac{1}{2}\Big[d \log(2\pi) + \log\det\Sigma + (x - \mu)^\top \Sigma^{-1} (x - \mu)\Big]$$
最後一項叫**馬氏距離**（Mahalanobis distance）的平方：用 $\Sigma$ 調整過的距離 —— 在 $\Sigma$「長」的方向上，走一樣遠算比較近。$\Sigma = \sigma^2 I$ 時退化成歐氏距離 / $\sigma^2$。
例（$d = 1$）：就是第六章前置複習 8 的 $-\frac{1}{2}\log(2\pi\sigma^2) - \frac{(x - \mu)^2}{2\sigma^2}$。

**3. Cholesky 分解**：對稱正定的 $\Sigma = L L^\top$，$L$ 是下三角。好處：
- $\log\det\Sigma = 2\sum_j \log L_{jj}$（不用算行列式，不會 overflow）；
- 馬氏距離 $= \lVert L^{-1}(x - \mu)\rVert^2$：解 $L z = (x - \mu)$（`np.linalg.solve` 或三角求解），不用算 $\Sigma^{-1}$。
例：$\Sigma = \begin{pmatrix}4 & 2\\ 2 & 2\end{pmatrix}$ → $L = \begin{pmatrix}2 & 0\\ 1 & 1\end{pmatrix}$，$\det\Sigma = 4 = (2 \cdot 1)^2$ ✓。

**4. 貝氏定理與 log-sum-exp**（第六章的重述）：
$$r_{ik} = \frac{\pi_k \mathcal{N}(x_i; \mu_k, \Sigma_k)}{\sum_{j} \pi_j \mathcal{N}(x_i; \mu_j, \Sigma_j)} \;\Rightarrow\; \log r_{ik} = a_{ik} - \text{logsumexp}_j(a_{ij}), \quad a_{ik} = \log\pi_k + \log\mathcal{N}(x_i; \mu_k, \Sigma_k)$$

**5. 加權的平均與共變異數**：權重 $r_i \ge 0$：$\mu = \frac{\sum_i r_i x_i}{\sum_i r_i}$、$\Sigma = \frac{\sum_i r_i (x_i - \mu)(x_i - \mu)^\top}{\sum_i r_i}$。所有 $r_i = 1$ 時就是一般的平均與共變異數。

**6. Jensen 不等式**（第五章前置複習 5 的重述）：$\log$ 是凹函數，$\log \sum_k q_k \frac{p_k}{q_k} \ge \sum_k q_k \log\frac{p_k}{q_k}$（$q$ 是機率分布）。EM 的保證就是從這裡來的（TODO 7）。

**7. 參數個數**（BIC 用）：一個 $d \times d$ 對稱矩陣有 $d(d+1)/2$ 個自由參數。例：$d = 4$ → 10。

#### 程式

**8. `np.linalg.cholesky(S)`** → $L$；`np.linalg.solve(L, B)` 解 $LZ = B$。
**9. `R.T @ X`**（第六章 TODO 3）：$(K, n) @ (n, d) \to (K, d)$，第 $k$ 列 $= \sum_i r_{ik} x_i$。
**10. 加權外積的總和**：$\sum_i r_i (x_i - \mu)(x_i - \mu)^\top$ = `(r[:, None] * diff).T @ diff`，`diff = X - mu` 是 $(n, d)$。
**11. `scipy.special.logsumexp(A, axis=1)`**（第六章）。

---
## 1. 資料與探索性資料分析（EDA）

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_iris

X_df, y = load_iris()
X_df = pd.DataFrame(X_df)
X = X_df.to_numpy(dtype=float)
y = np.asarray(y).astype(int)
FEATURES = X_df.columns.tolist()
n, d = X.shape
print("X：", X.shape, "；每一類：", np.bincount(y).tolist(), "；特徵：", FEATURES)
sns.pairplot(X_df.assign(species=pd.Categorical(y)), hue="species", plot_kws={"s": 12}, height=2.0)
plt.show()

In [ ]:
# 每一類的共變異數與相關
# 觀察：（三類的「形狀」一樣嗎？哪一類最小、最集中？versicolor 和 virginica 在哪幾個特徵上重疊？
#        類內的特徵之間有相關嗎（橢圓是斜的嗎）？K-Means 的「每群都是一樣大的圓」符合嗎？）
for k in range(3):
    Xk = X[y == k]
    print(f"類別 {k}：類內標準差 {Xk.std(axis=0).round(3)}；類內相關 petal length × petal width = {np.corrcoef(Xk[:, 2], Xk[:, 3])[0, 1]:.2f}、sepal length × sepal width = {np.corrcoef(Xk[:, 0], Xk[:, 1])[0, 1]:.2f}")
print("全體標準差：", X.std(axis=0).round(3))

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 哪一類跟另外兩類完全分開？哪兩類重疊？在哪些特徵上重疊？
# 答：
#
# (2) 類內：
#     (a) 三類的標準差（大小）一樣嗎？
#     (b) 類內的 sepal length 與 sepal width 相關嗎？這代表每一群在那兩個特徵上是「斜的橢圓」還是「正的圓」？
#     (c) 用 (a)(b) 說明：K-Means（每群是一樣大的圓）、Gaussian NB（第六章：每群是正的橢圓）、GMM full（每群是任意的橢圓）哪一個的假設最接近這份資料？
# 答：
#
# (3) 四個特徵單位都是 cm、尺度差不多，所以這一章不標準化。如果標準化，GMM full 的結果會變嗎？K-Means 呢？
#     （提示：GMM full 的馬氏距離會自動調整每個方向的尺度）
# 答：

---
## 2. Baseline

- **K-Means（$K = 3$）**（第十三章）。
- **Gaussian NB（監督，對照）**：用真實標籤訓練、在同一份資料上預測 —— 不是分群，而是「如果知道標籤、而且假設每類是正的橢圓」的上限參考。
- **QDA（監督，對照）**：用真實標籤估計每類的完整共變異數 = 「知道標籤的 GMM full」。

#### 本節名詞

- **ARI**（第十三章）：分群與真實品種的一致程度。分群的編號是任意的（群 0 不一定是 setosa），ARI 不受編號影響。

In [ ]:
from sklearn.cluster import KMeans
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import adjusted_rand_score
from sklearn.naive_bayes import GaussianNB
from learning_map.metrics import clustering_metrics

km = KMeans(3, n_init=10, random_state=0).fit(X)
baseline = pd.DataFrame(
    {
        "KMeans K=3": clustering_metrics(X, km.labels_, y),
        "GaussianNB (supervised, ref)": clustering_metrics(X, GaussianNB().fit(X, y).predict(X), y),
        "QDA (supervised, ref)": clustering_metrics(X, QuadraticDiscriminantAnalysis().fit(X, y).predict(X), y),
    }
).T
display(baseline)
print("K-Means 各群 vs 真實品種：")
display(pd.crosstab(km.labels_, y, rownames=["cluster"], colnames=["species"]))

In [ ]:
# TODO 2（問答）
#
# (1) K-Means 的 ARI 是多少？交叉表上，錯誤集中在哪兩個品種之間？
# 答：
#
# (2) QDA 是「知道答案的 GMM full」。它的 ARI 給了 GMM 一個什麼樣的參考？GMM 有可能比 QDA 好嗎？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 多變量常態的 log 密度（Cholesky） | 3 |
| 3B | EM：E 步（責任，log-sum-exp）、M 步（加權平均與共變異數） | 4、5、6 |
| 3C | 實驗：log-likelihood、四種共變異數、初始化與退化、軟分群 | — |

### 3A. 多變量常態的 log 密度

前置複習 2、3：

$$\log\mathcal{N}(x; \mu, \Sigma) = -\frac{1}{2}\Big[d\log(2\pi) + 2\sum_j \log L_{jj} + \lVert z\rVert^2\Big], \quad L z = x - \mu$$

一次處理全部 $n$ 個點：`diff = X - mu` 是 $(n, d)$；解 $L Z = \text{diff}^\top$ 得到 $(d, n)$ 的 $Z$，每一欄是一個點的 $z$；`(Z**2).sum(axis=0)` 就是每個點的馬氏距離平方。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $X$ | 資料 | $(n, d)$ | `X` |
| $\mu$、$\Sigma$ | 一個成分的平均、共變異數 | $(d,)$、$(d, d)$ | `mu`、`S` |
| $L$ | Cholesky 因子 | $(d, d)$ | `L` |
| $Z$ | $L^{-1}(X - \mu)^\top$ | $(d, n)$ | `Z` |
| maha | 馬氏距離平方 | $(n,)$ | `maha` |

#### 最小例子：TODO 3（log 密度）

$d = 2$，$\mu = (0, 0)$、$\Sigma = \begin{pmatrix}4 & 2\\ 2 & 2\end{pmatrix}$，兩個點 $(0, 0)$、$(2, 1)$。

1. **正確答案**：`scipy.stats.multivariate_normal.logpdf`，以及直接用 $\Sigma^{-1}$、$\det\Sigma$ 套公式。目標 $(-2.5310, -3.0310)$。
2. **積木**：`np.linalg.cholesky`、`np.linalg.solve(L, diff.T)`、$\log\det$ 兩種算法。
3. **留給你**：組成公式。

In [ ]:
# 最小例子：TODO 3（log 密度）
from scipy.stats import multivariate_normal

mu_mini = np.array([0.0, 0.0])
S_mini = np.array([[4.0, 2.0], [2.0, 2.0]])
X_mini = np.array([[0.0, 0.0], [2.0, 1.0]])

# ---- 1. 正確答案 ----
print("scipy：", multivariate_normal(mu_mini, S_mini).logpdf(X_mini).round(4))
direct = [-0.5 * (2 * np.log(2 * np.pi) + np.log(np.linalg.det(S_mini)) + (x - mu_mini) @ np.linalg.inv(S_mini) @ (x - mu_mini)) for x in X_mini]
print("直接套公式（inv、det）：", np.round(direct, 4))

# ---- 2. 積木 ----
L_mini = np.linalg.cholesky(S_mini)
print("\nL =\n", L_mini, "；L @ L.T 還原 Σ：", np.allclose(L_mini @ L_mini.T, S_mini))
print("log det Σ：np.log(det) =", round(np.log(np.linalg.det(S_mini)), 4), "；2·Σ log diag(L) =", round(2 * np.log(np.diag(L_mini)).sum(), 4))
Z_mini = np.linalg.solve(L_mini, (X_mini - mu_mini).T)
print("Z = solve(L, diff.T)", Z_mini.shape, "；(Z**2).sum(axis=0) =", (Z_mini**2).sum(axis=0), "← 馬氏距離平方")

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(gaussian_logpdf(X_mini, mu_mini, S_mini).round(4))

In [ ]:
def gaussian_logpdf(X: np.ndarray, mu: np.ndarray, S: np.ndarray) -> np.ndarray:
    """(n,)：log N(x_i; mu, S)，用 Cholesky（不算 inv、不算 det）。"""
    d = X.shape[1]
    L = np.linalg.cholesky(S)
    # TODO 3（程式）
    #   - Z = np.linalg.solve(L, (X − mu).T)      → (d, n)
    #   - maha = (Z**2).sum(axis=0)；logdet = 2·Σ log diag(L)
    #   - 回傳 −½ [d·log(2π) + logdet + maha]
    raise NotImplementedError("TODO 3：log 密度")


S_check = np.cov(X.T, bias=True)
print("✅ 與 scipy 的最大差異：", np.abs(gaussian_logpdf(X, X.mean(axis=0), S_check) - multivariate_normal(X.mean(axis=0), S_check).logpdf(X)).max(), "（應 < 1e-10）")

### 3B. EM 演算法

```
初始化 π, μ, Σ（本章：用 K-Means 的分群當成「硬」的責任，做一次 M 步）
重複：
    E 步：a_ik = log π_k + log N(x_i; μ_k, Σ_k)；ℓ_i = logsumexp_k a_ik；r_ik = exp(a_ik − ℓ_i)      ← TODO 4
          記錄平均 log-likelihood mean(ℓ_i)
    M 步：N_k = Σ_i r_ik；π_k = N_k / n；μ_k = Σ_i r_ik x_i / N_k                                  ← TODO 5
          Σ_k = Σ_i r_ik (x_i − μ_k)(x_i − μ_k)ᵀ / N_k + reg·I                                     ← TODO 6
    log-likelihood 的進步 < tol 就停
```

- **E 步**就是第六章的貝氏定理（每個成分 = 一個「類別」），只是「類別」不是給定的，是模型自己猜的。
- **M 步**就是第六章的「數數」，只是每個點對每一群的貢獻是 $r_{ik}$（0–1 之間），不是 0 或 1。
- **`reg_covar`**：對角線加一點點（$10^{-6}$），避免某一群的共變異數變成奇異矩陣（例如只剩一個點、或所有點在一條線上 → $\det\Sigma = 0$、$\log\det = -\infty$、likelihood 爆到無限大）。sklearn 也這樣做。

**跟 K-Means 的關係**：如果強迫 $\Sigma_k = \sigma^2 I$（全部一樣的圓）、$\pi_k$ 相等，再讓 $\sigma \to 0$，責任 $r_{ik}$ 會變成「最近的中心 = 1、其他 = 0」—— E 步變成 K-Means 的分配、M 步變成 K-Means 的更新。K-Means 是 GMM 的「硬」極限。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $a$ | $\log\pi_k + \log\mathcal{N}$ | $(n, K)$ | `log_prob` |
| $\ell_i$ | 每個點的 log-likelihood | $(n,)$ | `log_norm` |
| $R$ | 責任 | $(n, K)$ | `R` |
| $N_k$ | 有效點數 | $(K,)$ | `Nk` |
| `weights_`、`means_`、`covariances_` | $\pi$、$\mu$、$\Sigma$ | $(K,)$、$(K, d)$、$(K, d, d)$ | 屬性（名稱同 sklearn） |
| `log_likelihood_history_` | 每一次 E 步的平均 log-likelihood | list | 屬性 |

#### 程式結構圖

```
gmm = GMMScratch(n_components=3, init_resp=None 或 (n, K), tol=1e-10, reg_covar=1e-6).fit(X)
├── _init(X)：init_resp（沒給就用 K-Means 的硬分群）→ _m_step
├── 迴圈：R, ll = _e_step(X)   ← TODO 4（用 gaussian_logpdf）
│        _m_step(X, R)        ← TODO 5、6
└── fit 後：weights_、means_、covariances_、n_iter_、log_likelihood_history_；predict_proba(X)、predict(X)、score(X)
```

#### 最小例子：TODO 4、5、6（一輪 EM）

一維、4 個點 $x = (0, 1, 9, 10)$，$K = 2$，目前 $\pi = (0.5, 0.5)$、$\mu = (2, 8)$、$\sigma^2 = (4, 4)$。

1. **正確答案**：
   - E 步：$x = 0$ 到 $\mu_0$ 的 $(x - \mu)^2/2\sigma^2 = 0.5$、到 $\mu_1$ 是 8 → $r_{00} = 1/(1 + e^{-7.5}) \approx 0.99945$。四個點都幾乎完全屬於離自己近的那一群。
   - M 步：$N \approx (2, 2)$、$\mu \approx (0.51, 9.49)$、$\sigma^2 \approx (0.36, 0.36)$ —— 如果只用自己那兩個點會是 0.25；另一群的點只有 0.1%–0.2% 的責任，但它們離得很遠（距離平方約 80），還是把 $\sigma^2$ 撐大了一些。
2. **積木**：`logsumexp(..., axis=1, keepdims=True)`、`R.T @ X`、加權外積。
3. **留給你**：寫進 `_e_step`、`_m_step`。

In [ ]:
# 最小例子：TODO 4、5、6（一輪 EM）
from scipy.special import logsumexp

x1 = np.array([[0.0], [1.0], [9.0], [10.0]])
pi_m, mu_m, var_m = np.array([0.5, 0.5]), np.array([[2.0], [8.0]]), np.array([4.0, 4.0])

# ---- 1. 正確答案：一個點一個點 ----
R_loop = []
for x in x1[:, 0]:
    p = [pi_m[k] * np.exp(-((x - mu_m[k, 0]) ** 2) / (2 * var_m[k])) / np.sqrt(2 * np.pi * var_m[k]) for k in range(2)]
    R_loop.append([pk / sum(p) for pk in p])
R_loop = np.array(R_loop)
print("責任 R =\n", R_loop.round(5))
Nk_loop = R_loop.sum(axis=0)
mu_new = [(R_loop[:, k] * x1[:, 0]).sum() / Nk_loop[k] for k in range(2)]
var_new = [(R_loop[:, k] * (x1[:, 0] - mu_new[k]) ** 2).sum() / Nk_loop[k] for k in range(2)]
print("M 步：N_k =", Nk_loop.round(4), " μ =", np.round(mu_new, 4), " σ² =", np.round(var_new, 4))

# ---- 2. 積木 ----
a_mini = np.log(pi_m) + np.column_stack([-0.5 * (np.log(2 * np.pi * var_m[k]) + (x1[:, 0] - mu_m[k, 0]) ** 2 / var_m[k]) for k in range(2)])
print("\na = log π + log N：\n", a_mini.round(3))
print("logsumexp(a, axis=1, keepdims=True)：", logsumexp(a_mini, axis=1, keepdims=True).ravel().round(3))
print("R.T @ X（每群的加權總和）：", (R_loop.T @ x1).ravel().round(4))
diff0 = x1 - np.array(mu_new[0])
print("(r[:, None] * diff).T @ diff（第 0 群的加權平方和）：", ((R_loop[:, 0, None] * diff0).T @ diff0).ravel().round(4))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（給 1×1 的共變異數；reg_covar=0 才會跟上面一模一樣）：
# g_mini = GMMScratch(2, reg_covar=0.0); g_mini.weights_, g_mini.means_, g_mini.covariances_ = pi_m, mu_m, var_m[:, None, None]
# R_m, _ = g_mini._e_step(x1); print(R_m.round(5)); g_mini._m_step(x1, R_m); print(g_mini.means_.ravel().round(4), g_mini.covariances_.ravel().round(4))

In [ ]:
from typing import Self

from numpy.typing import ArrayLike


class GMMScratch:
    """高斯混合模型（full 共變異數），EM 演算法。屬性名稱同 sklearn GaussianMixture。"""

    def __init__(self, n_components: int = 3, init_resp: np.ndarray | None = None, max_iter: int = 1000, tol: float = 1e-10, reg_covar: float = 1e-6, random_state: int = 0) -> None:
        self.n_components = n_components
        self.init_resp = init_resp
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.random_state = random_state

    def _e_step(self, X: np.ndarray) -> tuple[np.ndarray, float]:
        """回傳 (R (n, K), 平均 log-likelihood)。"""
        K = self.n_components
        log_prob = np.column_stack([np.log(self.weights_[k]) + gaussian_logpdf(X, self.means_[k], self.covariances_[k]) for k in range(K)])  # (n, K)
        # TODO 4（程式）：E 步
        #   - log_norm = logsumexp(log_prob, axis=1, keepdims=True)   → (n, 1)：每個點的 log-likelihood
        #   - R = exp(log_prob − log_norm)；平均 log-likelihood = log_norm 的平均
        raise NotImplementedError("TODO 4：E 步")

    def _m_step(self, X: np.ndarray, R: np.ndarray) -> None:
        """用責任 R (n, K) 更新 weights_、means_、covariances_（就地）。"""
        n, d = X.shape
        Nk = R.sum(axis=0)  # (K,)
        # TODO 5（程式）：權重與平均
        #   - weights_ = Nk / n
        #   - means_ = (R.T @ X) / Nk[:, None]                         → (K, d)
        raise NotImplementedError("TODO 5：權重與平均")
        covs = []
        for k in range(self.n_components):
            diff = X - self.means_[k]  # (n, d)
            # TODO 6（程式）：第 k 群的共變異數
            #   - Σ_k = (R[:, k, None] * diff).T @ diff / Nk[k]  + reg_covar · I
            raise NotImplementedError("TODO 6：共變異數")
        self.covariances_ = np.array(covs)

    def _init(self, X: np.ndarray) -> None:
        R = self.init_resp
        if R is None:
            labels = KMeans(self.n_components, n_init=1, random_state=self.random_state).fit_predict(X)
            R = np.eye(self.n_components)[labels]
        self._m_step(X, R)

    def fit(self, X: ArrayLike) -> Self:
        X = np.asarray(X, dtype=float)
        self._init(X)
        self.log_likelihood_history_ = []
        prev = -np.inf
        for it in range(1, self.max_iter + 1):
            R, ll = self._e_step(X)
            self.log_likelihood_history_.append(ll)
            self._m_step(X, R)
            if abs(ll - prev) < self.tol:
                break
            prev = ll
        self.n_iter_ = it
        return self

    def predict_proba(self, X: ArrayLike) -> np.ndarray:
        return self._e_step(np.asarray(X, dtype=float))[0]

    def predict(self, X: ArrayLike) -> np.ndarray:
        return self.predict_proba(X).argmax(axis=1)

    def score(self, X: ArrayLike) -> float:
        """平均 log-likelihood（同 sklearn 的 score）。"""
        return self._e_step(np.asarray(X, dtype=float))[1]

In [ ]:
from sklearn.mixture import GaussianMixture

# 同樣的初始值：K-Means 的硬分群做一次 M 步 → 交給 sklearn 當 weights_init／means_init／precisions_init
R0 = np.eye(3)[km.labels_]
mine = GMMScratch(3, init_resp=R0).fit(X)
init = GMMScratch(3, init_resp=R0)
init._init(X)
sk = GaussianMixture(3, covariance_type="full", weights_init=init.weights_, means_init=init.means_, precisions_init=np.linalg.inv(init.covariances_), tol=1e-10, max_iter=1000).fit(X)
print(f"迭代次數：手寫 {mine.n_iter_}、sklearn {sk.n_iter_}")
print("✅ means_ 最大差異", np.abs(mine.means_ - sk.means_).max(), "；covariances_", np.abs(mine.covariances_ - sk.covariances_).max(), "；weights_", np.abs(mine.weights_ - sk.weights_).max(), "（都應 < 1e-8）")
print("✅ predict_proba 最大差異", np.abs(mine.predict_proba(X) - sk.predict_proba(X)).max(), "（應 < 1e-8）")
print("✅ 平均 log-likelihood：手寫 score", round(mine.score(X), 8), "vs sklearn score", round(sk.score(X), 8))
print("✅ log-likelihood 每一輪都不下降：", bool(np.all(np.diff(mine.log_likelihood_history_) >= -1e-12)))
print("ARI：GMM", round(adjusted_rand_score(y, mine.predict(X)), 4), "；K-Means", round(adjusted_rand_score(y, km.labels_), 4))

In [ ]:
# TODO 7（問答）EM
#
# (1) 最小例子：
#     (a) E 步之後，x = 1 屬於第 0 群的機率是多少？為什麼不是剛好 1？
#     (b) M 步之後 σ² 從 4 變成多少？為什麼一下子小這麼多？如果一開始 σ² = 100 呢？（提示：太大的 σ² → 責任會很平均）
# 答：
#
# (2) log-likelihood 為什麼不會下降？（鷹架）
#     (a) 對每個點，log Σ_k π_k N_k = log Σ_k r_k · (π_k N_k / r_k)。用前置複習 6（Jensen）寫出一個「下界」。
#     (b) 這個下界在 r_k = 目前的責任時「剛好等於」log-likelihood（E 步做的事）。
#     (c) M 步讓這個下界變大（因為它對 π、μ、Σ 是「加權版的 MLE」—— 前置複習 5）。所以 log-likelihood ≥ 新的下界 ≥ 舊的下界 = 舊的 log-likelihood。
#     用自己的話把 (a)–(c) 串起來；跟第十三章 K-Means「兩步都不會讓 J 變大」比較。
# 答：
#
# (3) GMM 的 ARI 比 K-Means 高多少？用 EDA 的觀察（各群的大小、斜的橢圓）解釋。
# 答：

### 3C. 實驗：log-likelihood、四種共變異數、初始化與退化、軟分群

#### 本節名詞

| `covariance_type` | 每一群的 $\Sigma_k$ | 形狀 | 參數個數（$\Sigma$ 部分） |
| --- | --- | --- | --- |
| `full` | 各自任意 | 各自任意方向、大小的橢圓 | $K \cdot d(d+1)/2$ |
| `tied` | 所有群共用一個 | 同樣形狀的橢圓（只是中心不同；= LDA 的假設） | $d(d+1)/2$ |
| `diag` | 各自、只有對角線 | 軸對齊的橢圓（= Gaussian NB 的假設） | $K \cdot d$ |
| `spherical` | 各自、$\sigma_k^2 I$ | 各自大小的圓 | $K$ |

**退化（degeneracy）**：某一群塌縮到一個點上，$\sigma \to 0$ → 那個點的密度 → ∞ → log-likelihood → ∞。不是好模型，是數值上的「作弊」；`reg_covar` 就是用來擋這個的。

In [ ]:
# 實驗 1：log-likelihood 的軌跡（隨機初始化 10 次）
from matplotlib.patches import Ellipse

fig, ax = plt.subplots(figsize=(8, 4))
finals = []
for s in range(10):
    rng = np.random.default_rng(s)
    R_rand = rng.dirichlet(np.ones(3), size=n)  # 隨機的軟分群當初始值
    g = GMMScratch(3, init_resp=R_rand).fit(X)
    ax.plot(g.log_likelihood_history_, alpha=0.7)
    finals.append((round(g.log_likelihood_history_[-1], 4), round(adjusted_rand_score(y, g.predict(X)), 3), g.n_iter_))
ax.set_xlabel("EM iteration")
ax.set_ylabel("mean log-likelihood")
ax.set_xscale("log")
ax.set_title("EM from 10 random initializations")
plt.show()
display(pd.DataFrame(finals, columns=["final mean log-lik", "ARI", "iterations"]).value_counts().rename("count").to_frame())

In [ ]:
# 實驗 2：四種共變異數（只用 petal length × sepal width 兩個特徵，方便畫橢圓）
def draw_ellipses(ax, means, covs, color="k"):
    for m, S in zip(means, covs):
        vals, vecs = np.linalg.eigh(S)
        angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
        for nsig in [1, 2]:
            ax.add_patch(Ellipse(m, 2 * nsig * np.sqrt(vals[-1]), 2 * nsig * np.sqrt(vals[0]), angle=angle, fill=False, color=color, lw=1, alpha=0.8))


cols = [FEATURES.index(c) for c in FEATURES if "petal length" in c or "sepal width" in c]
X2 = X[:, cols]
fig, axes = plt.subplots(1, 5, figsize=(24, 4.3))
for ax, ct in zip(axes, ["KMeans", "full", "tied", "diag", "spherical"]):
    if ct == "KMeans":
        lab = KMeans(3, n_init=10, random_state=0).fit_predict(X2)
        ax.scatter(X2[:, 0], X2[:, 1], c=lab, cmap="viridis", s=10)
        ax.set_title(f"K-Means: ARI {adjusted_rand_score(y, lab):.2f}")
        continue
    g = GaussianMixture(3, covariance_type=ct, n_init=5, random_state=0).fit(X2)
    if ct == "full":
        covs = g.covariances_
    elif ct == "tied":
        covs = [g.covariances_] * 3
    elif ct == "diag":
        covs = [np.diag(c) for c in g.covariances_]
    else:  # spherical：每一群一個純量
        covs = [c * np.eye(2) for c in g.covariances_]
    lab = g.predict(X2)
    ax.scatter(X2[:, 0], X2[:, 1], c=lab, cmap="viridis", s=10)
    draw_ellipses(ax, g.means_, covs)
    ax.set_title(f"{ct}: ARI {adjusted_rand_score(y, lab):.2f}, BIC {g.bic(X2):.0f}")
for ax in axes:
    ax.set_xlabel(FEATURES[cols[0]])
    ax.set_ylabel(FEATURES[cols[1]])
plt.show()

In [ ]:
# 實驗 3：退化。在資料裡放 3 個完全相同的點，reg_covar 從 0 到 1e-2
X_deg = np.vstack([X, np.repeat(X[:1], 3, axis=0)])
rows = []
for reg in [0.0, 1e-10, 1e-6, 1e-3, 1e-2]:
    R_init = np.zeros((len(X_deg), 4))
    R_init[:, :3] = np.eye(3)[np.r_[km.labels_, [km.labels_[0]] * 3]]
    R_init[-3:] = [0, 0, 0, 1]  # 第 4 群只包含那 3 個重複點
    try:
        with np.errstate(all="ignore"):
            g = GMMScratch(4, init_resp=R_init, reg_covar=reg, max_iter=200).fit(X_deg)
        rows.append({"reg_covar": reg, "final mean log-lik": g.log_likelihood_history_[-1], "smallest eigenvalue of any Σ": min(np.linalg.eigvalsh(c).min() for c in g.covariances_), "status": "ok"})
    except np.linalg.LinAlgError as err:
        rows.append({"reg_covar": reg, "final mean log-lik": np.nan, "smallest eigenvalue of any Σ": np.nan, "status": f"LinAlgError: {str(err)[:40]}"})
display(pd.DataFrame(rows).set_index("reg_covar"))

In [ ]:
# 實驗 4：軟分群 —— 不確定的點在哪裡？
g_full = GMMScratch(3, init_resp=R0).fit(X)
P = g_full.predict_proba(X)
maxp = P.max(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sc = axes[0].scatter(X[:, 2], X[:, 3], c=maxp, cmap="magma_r", s=25, vmin=0.5, vmax=1)
plt.colorbar(sc, ax=axes[0], label="max responsibility")
axes[0].set_xlabel(FEATURES[2])
axes[0].set_ylabel(FEATURES[3])
axes[0].set_title("How sure is the GMM about each point?")
axes[1].hist(maxp, bins=20)
axes[1].set_xlabel("max responsibility")
axes[1].set_title("Distribution of max responsibility")
plt.show()
mapping = {k: np.bincount(y[g_full.predict(X) == k], minlength=3).argmax() for k in range(3)}
pred_species = np.array([mapping[k] for k in g_full.predict(X)])
unsure = maxp < 0.9
print(f"max responsibility < 0.9 的點：{unsure.sum()} 個；其中分錯品種的：{int((pred_species[unsure] != y[unsure]).sum())}")
print(f"max responsibility ≥ 0.9 的點：{(~unsure).sum()} 個；其中分錯品種的：{int((pred_species[~unsure] != y[~unsure]).sum())}")

In [ ]:
# TODO 8（問答）
#
# (1) 實驗 1：10 次隨機初始化，最後的 log-likelihood 都一樣嗎？有幾種不同的結果？ARI 呢？
#     log-likelihood 比較高的那個解，ARI 一定比較高嗎？
# 答：
#
# (2) 實驗 2：
#     (a) 四種共變異數的橢圓長什麼樣子？哪一種 ARI 最高？哪一種 BIC 最低？兩者一致嗎？
#     (b) diag 對應第六章的哪個模型？tied 呢？（提示：tied = 所有類別共用一個共變異數 → 邊界是直線）
#     (c) full 的參數最多。資料很少、維度很高時，full 會有什麼問題？
# 答：
#
# (3) 實驗 3：
#     (a) reg_covar = 0 時發生什麼事？log-likelihood 變成多少（或出錯）？為什麼？
#     (b) 「log-likelihood 越高越好」在這裡還成立嗎？這說明 GMM 的最大概似估計有什麼根本的問題？
# 答：
#
# (4) 實驗 4：不確定的點（max responsibility < 0.9）落在哪裡？它們分錯的比例跟確定的點比如何？
#     K-Means 能給你這個資訊嗎？
# 答：

---
## 4. 套件實作：`GaussianMixture`、`BayesianGaussianMixture`

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `init_params` | 初始化方法：`"kmeans"`（預設）、`"k-means++"`、`"random"`、`"random_from_data"` |
| `n_init` | 跑幾次初始化，留 log-likelihood 最高的 |
| `lower_bound_` | 最後一次 E 步的平均 log-likelihood（EM 的下界） |
| `bic(X)`、`aic(X)` | 資訊準則：$-2\log L + p\log n$、$-2\log L + 2p$ |
| `BayesianGaussianMixture` | 給權重一個 Dirichlet 先驗；`weight_concentration_prior` 小時，用不到的成分權重會被壓到接近 0 → 給一個「上限」的 $K$，讓模型自己決定實際用幾個 |

In [ ]:
from sklearn.mixture import BayesianGaussianMixture

g_default = GaussianMixture(3, random_state=0).fit(X)
print("GaussianMixture 預設：ARI", round(adjusted_rand_score(y, g_default.predict(X)), 4), "；n_iter", g_default.n_iter_, "；lower_bound_", round(g_default.lower_bound_, 4))
rows = []
for prior in [1e-3, 1e-1, 1.0, 10.0]:
    b = BayesianGaussianMixture(n_components=8, weight_concentration_prior=prior, max_iter=2000, random_state=0).fit(X)
    rows.append({"weight_concentration_prior": prior, "weights (sorted)": np.sort(b.weights_)[::-1].round(3).tolist(), "components with weight > 0.02": int((b.weights_ > 0.02).sum()), "ARI": adjusted_rand_score(y, b.predict(X))})
display(pd.DataFrame(rows).set_index("weight_concentration_prior"))

In [ ]:
# TODO 9（問答）
#
# (1) BayesianGaussianMixture 給了 8 個成分的上限。prior 很小時，實際用到幾個？prior 變大呢？
#     這跟「直接指定 K」比，好處和壞處是什麼？
# 答：
#
# (2) GaussianMixture 的預設初始化是 K-Means。為什麼用 K-Means 初始化通常比隨機好？（實驗 1）
# 答：

---
## 5. 選成分數與共變異數形式：BIC

$$\text{BIC} = -2 \log L + p \log n, \qquad \log L = n \cdot \ell \;(\text{總 log-likelihood})$$

- 第一項：模型多貼近資料（越小越好）；第二項：參數越多罰越多（$\log n$ 倍）。
- $p$（full、$K$ 個成分、$d$ 維）：權重 $K - 1$（總和 1，少一個自由度）＋ 平均 $Kd$ ＋ 共變異數 $K \cdot d(d+1)/2$。

#### 符號表（BIC）

| 符號 | 意義 | 程式裡 |
| --- | --- | --- |
| $\ell$ | 平均 log-likelihood（`score`） | `mean_ll` |
| $p$ | 參數個數，依 `covariance_type` 不同 | `n_parameters(...)` |

#### 最小例子：TODO 10（參數個數）

$K = 2$、$d = 3$：

| 形式 | 權重 | 平均 | 共變異數 | 合計 |
| --- | --- | --- | --- | --- |
| full | 1 | 6 | $2 \times 6 = 12$ | 19 |
| tied | 1 | 6 | 6 | 13 |
| diag | 1 | 6 | $2 \times 3 = 6$ | 13 |
| spherical | 1 | 6 | 2 | 9 |

把表寫成程式，跟 sklearn 的 `_n_parameters()` 對照。

In [ ]:
def n_parameters(K: int, d: int, covariance_type: str) -> int:
    """GMM 的自由參數個數。"""
    # TODO 10（程式）
    #   - 權重 K − 1、平均 K·d
    #   - 共變異數：full K·d(d+1)/2、tied d(d+1)/2、diag K·d、spherical K
    raise NotImplementedError("TODO 10：參數個數")


def bic_scratch(mean_ll: float, n: int, p: int) -> float:
    return -2 * n * mean_ll + p * np.log(n)


for ct in ["full", "tied", "diag", "spherical"]:
    g = GaussianMixture(3, covariance_type=ct, random_state=0).fit(X)
    print(f"✅ {ct:<9} 參數個數：手寫 {n_parameters(3, d, ct)}、sklearn {g._n_parameters()}；BIC 差 {abs(bic_scratch(g.score(X), n, n_parameters(3, d, ct)) - g.bic(X)):.1e}")

rows = []
for ct in ["full", "tied", "diag", "spherical"]:
    for K in range(1, 7):
        g = GaussianMixture(K, covariance_type=ct, n_init=5, random_state=0).fit(X)
        rows.append({"covariance_type": ct, "K": K, "BIC": g.bic(X), "ARI": adjusted_rand_score(y, g.predict(X)) if K > 1 else 0.0})
bic_tbl = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ct, sub in bic_tbl.groupby("covariance_type"):
    axes[0].plot(sub["K"], sub["BIC"], "o-", label=ct)
    axes[1].plot(sub["K"], sub["ARI"], "o-", label=ct)
axes[0].set_title("BIC (lower is better)")
axes[1].set_title("ARI vs species (not available in real life)")
for ax in axes:
    ax.set_xlabel("K")
    ax.legend()
plt.show()
best_bic = bic_tbl.loc[bic_tbl["BIC"].idxmin()]
print("BIC 最小：", best_bic.to_dict())
display(bic_tbl.pivot_table(index="K", columns="covariance_type", values="BIC").round(1))

In [ ]:
# TODO 11（問答）BIC
#
# (1) BIC 最小的組合是什麼？K 是多少？跟真實的品種數 3 一樣嗎？
# 答：
#
# (2) 如果 BIC 選了 K = 2：用 EDA 解釋為什麼「兩群」在 likelihood 上可能比「三群」划算。
#     （提示：versicolor 和 virginica 重疊很多 —— 用一個大的橢圓包住兩者，跟用兩個橢圓分開，likelihood 差多少？多出來的參數值得嗎？）
#     那「品種」是資料本身的結構，還是人類給的標籤？非監督學習能保證找到人類想要的分法嗎？
# 答：
#
# (3) ARI 曲線（真實應用中沒有）在 K = 3 時哪一種共變異數最高？如果你知道「應該是 3 群」，你會選哪一種？
# 答：

---
## 6. 評估

`clustering_metrics(X, labels, y)`；`log_result`／`compare` 寫進 `results/iris_clustering.csv`，依 ARI 排序。

In [ ]:
from learning_map.results import compare, log_result

BENCH = "iris_clustering"
gb = GaussianMixture(int(best_bic["K"]), covariance_type=best_bic["covariance_type"], n_init=5, random_state=0).fit(X)
runs = {
    "KMeans K=3": (km.labels_, {"K": 3}),
    "GMM full K=3 (scratch)": (mine.predict(X), {"K": 3, "covariance_type": "full"}),
    "GMM full K=3 (sklearn)": (GaussianMixture(3, n_init=5, random_state=0).fit_predict(X), {"K": 3, "covariance_type": "full"}),
    "GMM diag K=3 (sklearn)": (GaussianMixture(3, covariance_type="diag", n_init=5, random_state=0).fit_predict(X), {"K": 3, "covariance_type": "diag"}),
    f"GMM BIC-best (sklearn)": (gb.predict(X), {"K": int(best_bic["K"]), "covariance_type": best_bic["covariance_type"]}),
}
scores = {}
for name, (lab, params) in runs.items():
    m = clustering_metrics(X, lab, y)
    scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(scores).T)
compare(BENCH, "ari")

---
## 7. 診斷

GMM（full、$K = 3$）分錯的花：它們的責任、到各成分的馬氏距離。

In [ ]:
lab = mine.predict(X)
mapping = {k: np.bincount(y[lab == k], minlength=3).argmax() for k in range(3)}
pred_species = np.array([mapping[k] for k in lab])
display(pd.crosstab(y, pred_species, rownames=["true species"], colnames=["GMM (mapped)"]))
wrong = np.flatnonzero(pred_species != y)
maha = np.column_stack([-2 * gaussian_logpdf(X, mine.means_[k], mine.covariances_[k]) - d * np.log(2 * np.pi) - np.linalg.slogdet(mine.covariances_[k])[1] for k in range(3)])
display(pd.DataFrame({"true": y[wrong], "pred": pred_species[wrong], "max resp": P[wrong].max(axis=1).round(3), **{f"maha² to comp {k}": maha[wrong, k].round(2) for k in range(3)}}, index=wrong))
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X[:, 2], X[:, 3], c=pred_species, cmap="viridis", s=15, alpha=0.6)
ax.scatter(X[wrong, 2], X[wrong, 3], facecolors="none", edgecolors="r", s=90, label="misclustered")
draw_ellipses(ax, mine.means_[:, 2:], mine.covariances_[:, 2:, 2:])
ax.set_xlabel(FEATURES[2])
ax.set_ylabel(FEATURES[3])
ax.legend()
plt.show()

In [ ]:
# TODO 12（問答，本章總結）
#
# (1) 分錯的花：
#     (a) 在哪兩個品種之間？它們的 max responsibility 大約多少 —— 模型知道自己沒把握嗎？
#     (b) 看 maha²：分錯的點到「真實品種那個成分」的馬氏距離，跟到「被分到的成分」比如何？
#     (c) 這些錯誤可以靠換模型解決嗎？還是 4 個特徵本來就分不開？（對照第二節 QDA 的 ARI）
# 答：
#
# (2) 分群三部曲比較表（填空）：
#
#     |                    | K-Means（第十三章） | DBSCAN（第十四章） | GMM（本章） |
#     | 群的形狀           |                     |                    |             |
#     | 硬／軟             |                     |                    |             |
#     | 要指定群數嗎       |                     |                    |             |
#     | 雜訊點             |                     |                    |             |
#     | 演算法             |                     |                    |             |
#     | 選群數／參數的方法 |                     |                    |             |
#     | 能給新點機率嗎     |                     |                    |             |
#
# (3) GMM 也是一個「密度估計」：score_samples(x) 給每個點的 log 密度。可以怎麼拿來做異常偵測？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Iris（150 筆、4 個特徵；品種只用於評估）。

| 方法 | 做法 | 關鍵性質 | ARI | BIC |
| --- | --- | --- | --- | --- |
| K-Means $K = 3$ | | | | — |
| GMM full $K = 3$（手寫） | EM、Cholesky、log-sum-exp | | | |
| GMM diag／tied／spherical | | | | |
| GMM（BIC 選的） | | | | |
| QDA（監督，對照） | | | | — |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ EM 的一般形式：完整資料的 log-likelihood、Q 函數、ELBO；EM 是 MM（minorize–maximize）演算法
- ☐ GMM 的 likelihood 沒有上界（退化）；MAP 估計、先驗
- ☐ 變分推論（Bayesian GMM 的 variational EM）、Dirichlet process mixture
- ☐ K-Means 是 GMM 在 $\sigma \to 0$ 的極限（hard EM）

**延伸**
- ☐ 用 GMM 做密度估計、異常偵測、生成新樣本（`sample`）
- ☐ 其他用 EM 的模型：HMM（第 20 章的 Baum–Welch）、混合迴歸、缺值補值
- ☐ 高維時的 GMM：共變異數正則化、先降維（第 16 章 PCA）